In [14]:
import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats
from pathlib import Path

boundaries_path = "../data/raw/boundaries/geoBoundaries-MOZ-ADM2.geojson"
raster_path = "../data/raw/worldpop/moz_under_age_18_2019/moz_T_Under_18_2019_CN_100m_R2025A_v1.tif"

boundaries = gpd.read_file(boundaries_path)

stats = zonal_stats(
    boundaries,
    raster_path,
    stats=["sum", "count"],
    nodata=-99999,
)

stats_df = pd.DataFrame(stats)

print(stats_df.head())
print(stats_df.shape)
print(stats_df.isna().sum())

"""
====================================================================================
1. Identify the 2 districts with missing 'sum' values
====================================================================================
"""
# print("\n")
# results = boundaries.copy()
# results["under18_sum"] = stats_df["sum"]
# results["valid_cell_count"] = stats_df["count"]

# missing = results[results["under18_sum"].isna()]

# print(missing[["shapeName", "shapeID", "shapeType", "valid_cell_count", "under18_sum"]])

# print(missing["valid_cell_count"].describe())

"""
====================================================================================
2. See if cell-center rule is missing tiny islands
====================================================================================
"""
# print("\n")
# stats_all_touched = zonal_stats(
#     boundaries,
#     raster_path,
#     stats=["sum", "count"],
#     nodata=-99999,
#     all_touched=True,
# )

# stats_all_touched_df = pd.DataFrame(stats_all_touched)

# comparison = boundaries[["shapeName", "shapeID"]].copy()
# comparison["count_default"] = stats_df["count"]
# comparison["sum_default"] = stats_df["sum"]
# comparison["count_all_touched"] = stats_all_touched_df["count"]
# comparison["sum_all_touched"] = stats_all_touched_df["sum"]

# print(
#     comparison.loc[
#         comparison["shapeName"].isin(["Ilha Licom", "Ilha Risunodo"]),
#         [
#             "shapeName",
#             "count_default",
#             "sum_default",
#             "count_all_touched",
#             "sum_all_touched",
#         ],
#     ]
# )

"""
====================================================================================
3. Create a clean joined results table in memory
====================================================================================
"""
# print("\n")
# results = boundaries.copy()

# results["under18_sum"] = stats_df["sum"]
# results["valid_cell_count"] = stats_df["count"]
# results["worldpop_valid_cells"] = results["valid_cell_count"] > 0

# print(results[["shapeName", "under18_sum", "valid_cell_count", "worldpop_valid_cells"]].head())

# print(results["worldpop_valid_cells"].value_counts())

# print(results["under18_sum"].describe())

"""
====================================================================================
4. Double check the highest and lowest district values
====================================================================================
"""
print("\n")
print(
    results.loc[
        results["worldpop_valid_cells"],
        ["shapeName", "under18_sum", "valid_cell_count"]
    ]
    .sort_values("under18_sum", ascending=False)
    .head(10)
)

print(
    results.loc[
        results["worldpop_valid_cells"],
        ["shapeName", "under18_sum", "valid_cell_count"]
    ]
    .sort_values("under18_sum", ascending=True)
    .head(10)
)

"""
==============================================================================================
5. Save two versions of the data: spatial for geo and non-spatial for inspection/reporting
==============================================================================================
"""
print("\n")
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

gpkg_path = processed_dir / "moz_adm2_under18_2019.gpkg"
csv_path = processed_dir / "moz_adm2_under18_2019.csv"

results.to_file(gpkg_path, layer="moz_adm2_under18_2019", driver="GPKG")

results.drop(columns="geometry").to_csv(csv_path, index=False)

print(gpkg_path)
print(csv_path)

"""
====================================================================================
6. Confirm outputs
====================================================================================
"""
print("\n")
print(gpkg_path.exists(), gpkg_path.stat().st_size)
print(csv_path.exists(), csv_path.stat().st_size)


    count            sum
0  363972  213755.062500
1   51808   92980.007812
2   81424  204043.000000
3  180952  263586.812500
4   45246  103711.765625
(159, 2)
count    0
sum      2
dtype: int64


             shapeName    under18_sum  valid_cell_count
26    Cidade Da Matola  473215.625000             46310
30    Cidade De Maputo  442010.875000             32065
31   Cidade De Nampula  425966.875000             30442
108            Milange  363712.437500            351980
25     Cidade Da Beira  291995.781250             31719
3              Angonia  263586.812500            180952
48               Gurue  251897.593750            310634
112             Mocuba  242756.859375            320779
118             Monapo  234931.093750            157711
121         Morrumbala  231907.734375            250850
         shapeName   under18_sum  valid_cell_count
60     Lago Niassa   1746.807617              1167
50             Ibo   5953.359375              1586
18         Chigubo  10106.416016   